Description: This script will summarize haunted place counts, alchohol metrics, and most common time of sightings per city.
The cleaned data is exported as a JSON file to use for the D3 bubble map visualization. 


In [ ]:


import pandas as pd
import json
from collections import Counter

input_path = "/Users/rehamatai/dsci_550_a1/data/processed/haunted_places_features_added_v2.tab"
output_path = "/Users/rehamatai/dsci_550_a1/data/processed/bubble_map_data.json"

#load the data
df = pd.read_csv(input_path, sep="\t")

# clean the data
df["City"] = df["City"].astype(str).str.strip().str.replace(r'[<>"]', '', regex=True)

df["Total_Deaths"] = (
    df["Total_Deaths"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .astype(float)
)

df["Percent_Under_21"] = (
    df["Percent_Under_21"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .astype(float)
)

# drop missing values 
df = df.dropna(subset=[
    "City", "State_Abbrev", "Latitude", "Longitude",
    "Total_Deaths", "Percent_Under_21", "Time_of_Day"
])

# group by location 
grouped = df.groupby(["City", "State_Abbrev", "Latitude", "Longitude"]).agg(
    haunted_count=("Haunted_Places_Id", "count"),
    avg_total_deaths=("Total_Deaths", "mean"),
    avg_percent_under_21=("Percent_Under_21", "mean")
).reset_index()

# add most common time of day 
def most_common_time(group):
    return Counter(group["Time_of_Day"]).most_common(1)[0][0]

grouped["most_common_time_of_day"] = (
    df.groupby(["City", "State_Abbrev", "Latitude", "Longitude"])["Time_of_Day"]
    .agg(lambda x: Counter(x).most_common(1)[0][0])
    .reset_index(drop=True)

)

# round #s
grouped["avg_total_deaths"] = grouped["avg_total_deaths"].round(2)
grouped["avg_percent_under_21"] = grouped["avg_percent_under_21"].round(2)


grouped = grouped.rename(columns={
    "City": "city",
    "State_Abbrev": "state",
    "Latitude": "latitude",
    "Longitude": "longitude"
})


grouped.insert(0, "id", ["doc_" + str(i) for i in range(len(grouped))])

# export to json 
data = grouped.to_dict(orient="records")
with open(output_path, "w") as f:
    json.dump(data, f, indent=2)

print(f"JSON saved to: {output_path}")


In [11]:
# import json

# input_path = "/Users/rehamatai/dsci_550_a1/data/processed/bubble_map_data.json"
# output_path = "/Users/rehamatai/dsci_550_a1/data/processed/bubble_map_data_add_wrapped.json"

# with open(input_path) as f:
#     docs = json.load(f)

# wrapped_docs = [{"add": {"doc": doc}} for doc in docs]

# with open(output_path, "w") as f:
#     json.dump(wrapped_docs, f, indent=2)

# print(f"✅ Wrapped docs saved to {output_path}")


✅ Wrapped docs saved to /Users/rehamatai/dsci_550_a1/data/processed/bubble_map_data_add_wrapped.json
